In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from xgboost import XGBClassifier

import matplotlib.pyplot as plt

In [2]:
import numpy as np
import pandas as pd
pd.set_option('display.max_rows',None);
pd.set_option('display.max_columns',None)
train=pd.concat([pd.read_csv('/kaggle/input/datasets/shaadahmad51/aresole/SDNFlow_Dataset_Normal.csv'),pd.read_csv('/kaggle/input/datasets/shaadahmad51/aresole/SDNFlow_Dataset_attack.csv')])
tester=pd.read_csv('/kaggle/input/datasets/shaadahmad51/aresole/modified_concatenated_InSDN_DatasetCSV.csv')

In [3]:
train=train.drop(columns=['flow_ID','eth_type','ipv4_src','ipv4_dst','ip_proto','src_port','dst_port','TnBPDstIP','TnBPSrcIP','TnPPDstIP','TnPPSrcIP','TnPPerDport','Tn_FlowsPDstIP','Tn_FlowsPSrcIP','Attack'])

In [4]:
tester = tester.drop(columns=[
    'Pkt Len Min',
    'Pkt Len Max',
    'Pkt Len Mean',
    'Pkt Len Std',
    'Pkt Len Var',
    'FIN Flag Cnt',
    'SYN Flag Cnt',
    'RST Flag Cnt',
    'PSH Flag Cnt',
    'ACK Flag Cnt',
    'URG Flag Cnt',
    'CWE Flag Count',
    'ECE Flag Cnt',
    'Down/Up Ratio'
])

In [5]:
tester = tester.drop(columns=[
    'Flow IAT Mean',
    'Flow IAT Std',
    'Flow IAT Max',
    'Flow IAT Min',
    'Fwd IAT Tot',
    'Fwd IAT Mean',
    'Fwd IAT Std',
    'Fwd IAT Max',
    'Fwd IAT Min',
    'Bwd IAT Tot',
    'Bwd IAT Mean',
    'Bwd IAT Std',
    'Bwd IAT Max',
    'Bwd IAT Min',
    'Fwd PSH Flags',
    'Bwd PSH Flags',
    'Fwd URG Flags',
    'Bwd URG Flags',
    'Fwd Header Len',
    'Bwd Header Len'
])

In [6]:
tester = tester.drop(columns=[
    'Fwd Pkt Len Max',
    'Fwd Pkt Len Min',
    'Fwd Pkt Len Mean',
    'Fwd Pkt Len Std',
    'Bwd Pkt Len Max',
    'Bwd Pkt Len Min',
    'Bwd Pkt Len Mean',
    'Bwd Pkt Len Std'
])

In [7]:
tester=tester.drop(columns=['Flow ID','Timestamp','Flow Duration'])

tester['Tot Fwd Pkts']=tester['Tot Fwd Pkts']+tester['Tot Bwd Pkts']
tester=tester.rename(columns={'Tot Fwd Pkts':'tot_pkts'})

tester['TotLen Fwd Pkts']=tester['TotLen Fwd Pkts']+tester['TotLen Bwd Pkts']
tester=tester.rename(columns={'TotLen Fwd Pkts':'tot_len_pkts'})

tester=tester.drop(columns=['Tot Bwd Pkts','TotLen Bwd Pkts','Fwd Pkts/s','Bwd Pkts/s'])

In [8]:
tester=tester.rename(columns={'Flow Byts/s':'flow_byts_s','Flow Pkts/s':'flow_pkts_s','Pkt Size Avg':'pkt_size_average', 'Active Mean': 'active_mean',
    'Active Std': 'active_std',
    'Active Max': 'active_max',
    'Active Min': 'active_min',
    'Idle Mean': 'idle_mean',
    'Idle Std': 'idle_std',
    'Idle Max': 'idle_max',
    'Idle Min': 'idle_min'
})

In [9]:
train = train.drop(columns=[
    'fwd_header_len',
    'Byts_max_interval_2s',
    'Byts_min_interval_2s',
    'Byts_mean_interval_2s',
    'Byts_std_interval_2s',
    'Pkts_max_interval_2s',
    'Pkts_mean_interval_2s',
    'Pkts_min_interval_2s',
    'Pkts_std_interval_2s'
])

In [10]:
train=train.drop(columns='flow_duration')
train.head()
y=train['Category']
train.head()
X=train
y.head()
y=y.str.upper()
y.head()
y.value_counts()
y=y[~y.isin(['STREAMING','SSH','ICMP','FTP','NTP'])]
y.value_counts()
tester['Label'].value_counts()
tester['Label']=tester['Label'].str.upper()
tester['Label'].value_counts()
tester['Label']=tester['Label'].str.strip()
tester['Label'].value_counts()

Label
DDOS          121942
PROBE          98129
NORMAL         68424
DOS            53616
BFA             1405
WEB-ATTACK       192
BOTNET           164
U2R               17
Name: count, dtype: int64

In [11]:
y=y.replace({
    'HTTP':'NORMAL',
    'PASSWORD-GUESSING':'BFA',
    'DNS':'BOTNET'
})
y.value_counts()
y.isnull().sum()
X.isnull().sum()
from sklearn.model_selection import train_test_split
X.shape
y.shape
y.head()
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
y=le.fit_transform(y)
X.shape
y.shape

(659789,)

In [12]:
train['Category'].value_counts()
safe_train=train
train['Category']=train['Category'].str.strip()
train['Category']=train['Category'].str.upper()
train['Category'].value_counts()
train=train[~train['Category'].isin(['STREAMING','SSH','ICMP','FTP','NTP'])]
train['Category'].value_counts()
y=train['Category']
y=le.fit_transform(y)
df=pd.DataFrame(y)
df.value_counts()

0
3    164974
5    151456
2    123807
0     74644
4     67288
1     53551
6     17787
7      6282
Name: count, dtype: int64

In [13]:
X=train.drop(columns='Category')
X.shape
y.shape
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [14]:
label_encoder = LabelEncoder()

y_train_encoded = label_encoder.fit_transform(y_train)

print("Classes:")
print(label_encoder.classes_)

Classes:
[0 1 2 3 4 5 6 7]


In [15]:
class Generator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, output_dim)
        )

    def forward(self, z):
        return self.model(z)

In [16]:
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [17]:
latent_dim = 10
epochs = 20
batch_size = 32

X_np = np.asarray(X_train)

dataset = TensorDataset(
    torch.FloatTensor(X_np)
)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True
)

generator = Generator(
    latent_dim,
    X_np.shape[1]
)

discriminator = Discriminator(
    X_np.shape[1]
)

criterion = nn.BCELoss()

g_optimizer = optim.Adam(
    generator.parameters(),
    lr=0.0002
)

d_optimizer = optim.Adam(
    discriminator.parameters(),
    lr=0.0002
)

g_losses = []
d_losses = []

for epoch in range(epochs):

    epoch_g_loss = 0
    epoch_d_loss = 0

    for real_batch in loader:

        real_data = real_batch[0]

        batch_size_current = real_data.size(0)

        real_labels = torch.ones(
            batch_size_current,
            1
        )

        fake_labels = torch.zeros(
            batch_size_current,
            1
        )

        # -----------------------
        # Train Discriminator
        # -----------------------

        d_optimizer.zero_grad()

        real_output = discriminator(real_data)

        d_real_loss = criterion(
            real_output,
            real_labels
        )

        noise = torch.randn(
            batch_size_current,
            latent_dim
        )

        fake_data = generator(noise)

        fake_output = discriminator(
            fake_data.detach()
        )

        d_fake_loss = criterion(
            fake_output,
            fake_labels
        )

        d_loss = d_real_loss + d_fake_loss

        d_loss.backward()

        d_optimizer.step()

        # -----------------------
        # Train Generator
        # -----------------------

        g_optimizer.zero_grad()

        noise = torch.randn(
            batch_size_current,
            latent_dim
        )

        generated = generator(noise)

        prediction = discriminator(
            generated
        )

        g_loss = criterion(
            prediction,
            real_labels
        )

        g_loss.backward()

        g_optimizer.step()

        epoch_g_loss += g_loss.item()
        epoch_d_loss += d_loss.item()

    g_losses.append(
        epoch_g_loss / len(loader)
    )

    d_losses.append(
        epoch_d_loss / len(loader)
    )

    if epoch % 20 == 0:
        print(
            f"Epoch {epoch} | "
            f"G Loss={g_losses[-1]:.4f} | "
            f"D Loss={d_losses[-1]:.4f}"
        )

Epoch 0 | G Loss=1.1854 | D Loss=0.8805


In [18]:
majority_count = np.bincount(
    y_train_encoded
).max()

synthetic_X = []
synthetic_y = []

for cls in np.unique(y_train_encoded):

    current_count = np.sum(
        y_train_encoded == cls
    )

    needed = majority_count - current_count

    if needed > 0:

        noise = torch.randn(
            needed,
            latent_dim
        )

        fake_samples = generator(
            noise
        ).detach().numpy()

        synthetic_X.append(fake_samples)

        synthetic_y.append(
            np.full(
                needed,
                cls
            )
        )

X_balanced = np.vstack(
    [X_np] + synthetic_X
)

y_balanced = np.concatenate(
    [y_train_encoded] + synthetic_y
)

print("Balanced Shape:")
print(X_balanced.shape)

Balanced Shape:
(1055832, 13)


In [19]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_balanced,
    y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='multi:softprob',
    num_class=len(np.unique(y_balanced)),
    eval_metric='mlogloss',
    random_state=42
)

eval_set = [
    (X_tr, y_tr),
    (X_val, y_val)
]

model.fit(
    X_tr,
    y_tr,
    eval_set=eval_set,
    verbose=False
)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.05, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None, num_class=8, ...)

In [20]:
preds = model.predict(X_val)

accuracy = accuracy_score(
    y_val,
    preds
)

print(
    f"Accuracy: {accuracy:.4f}"
)

print(
    classification_report(
        y_val,
        preds
    )
)

Accuracy: 0.4686
              precision    recall  f1-score   support

           0       0.94      0.12      0.22     26396
           1       0.83      0.08      0.14     26395
           2       0.89      0.65      0.75     26396
           3       0.86      0.65      0.74     26396
           4       0.91      0.31      0.46     26396
           5       0.48      0.88      0.62     26396
           6       0.66      0.08      0.15     26396
           7       0.25      0.98      0.39     26396

    accuracy                           0.47    211167
   macro avg       0.73      0.47      0.43    211167
weighted avg       0.73      0.47      0.43    211167

